In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner = "DataTalksCLub",
    repo_name = "llm-zoomcamp",
    commit_id = "8c1834d",
    allowed_extensions = {"md"},
    filename_filter = lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
from embedder import Embedder

embed = Embedder()

In [3]:
# Q1. Embedding a query
q1 = "How does approximate nearest neighbor search work?"
vq1 = embed.encode(q1)
vq1[0]

np.float64(-0.02058203437252893)

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
# Q2. Cosine similarity
sqlitesearch = next((doc for doc in documents if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"))
vsqlitesearch = embed.encode(sqlitesearch["content"])
vsqlitesearch.dot(vq1)

np.float64(0.36107027225589694)

In [6]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)


In [7]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    batch_content = [chunk["content"] for chunk in batch]
    batch_vectors = embed.encode_batch(batch_content)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [8]:
scores = X.dot(vq1)

In [9]:
idx = np.argmax(scores)

In [22]:
# Q3. Chunking and search by hand
chunks[idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [11]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields = ["filename"])
vindex.fit(X, chunks);

In [12]:
# Q4. Vector search with minsearch
q2 = "What metric do we use to evaluate a search engine?"
vq2 = embed.encode(q2)

vindex.search(vq2, num_results=1)[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

In [13]:

from minsearch import Index

index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"]
)
index.fit(chunks)

In [14]:
q3 = "How do I store vectors in PostgreSQL?"
vq3 = embed.encode(q3)

In [15]:
[result["filename"] for result in vindex.search(vq3, num_results=5)]

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [16]:
[result["filename"] for result in index.search(q3, num_results=5)]
# Q5. Text search vs vector search
# Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?
# Answer is pgvector, which appear in vector search but not in text search

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

In [17]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [18]:
q4 = "How do I give the model access to tools?"
vq4 = embed.encode(q4)

In [19]:
text_results = index.search(q4, num_results=5)
vector_results = vindex.search(vq4, num_results=5)

In [20]:
results = rrf([vector_results, text_results])

In [21]:
# Q6. Hybrid search
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'